# Triton matmul on a T4: grouped ordering and autotuning

Companion to the card [Triton matmul: grouped ordering and autotuning](https://seyonv.github.io/explainers/perf-3-kernels/triton-matmul.html).

**Runtime → Change runtime type → T4 GPU**, then Run all. PyTorch and Triton come preinstalled on Colab.

The kernel is Triton's tutorial 03 (`python/tutorials/03-matrix-multiplication.py`, MIT License, © 2018-2020 Philippe Tillet, © 2020-2022 OpenAI), adapted: `tl.assume` hints removed so it runs on older Triton versions, the jit kernel kept separate from the autotuner so we can force one config, and CUDA only.

What you will measure (fp16 inputs, fp32 accumulation; the T4 has no BF16 or FP8 tensor cores):

1. Correctness against `torch.matmul` (cuBLAS), including a size that is not a multiple of any block (the K-tail mask) and the fused leaky_relu.
2. TFLOPS for Triton and `torch.matmul` at 1024², 2048², 4096², timed with `triton.testing.do_bench`, as a % of the T4's 65 TFLOPS FP16 tensor peak.
3. The config the autotuner picked for each shape.
4. `GROUP_SIZE_M` = 1 (row-major launch order) vs 2, 4, 8, 16 with everything else fixed, on a square 4096² and on the Llama-3.1-8B up-projection shape (4096 × 4096 @ 4096 × 14336).

Colab's T4 clocks vary (70 W power cap), so numbers move between sessions. Run the benchmark cells twice and trust the second run.

In [ ]:
!nvidia-smi

In [ ]:
import torch, triton, triton.language as tl
import triton.testing
print("torch", torch.__version__, "| triton", triton.__version__)
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0), "capability", torch.cuda.get_device_capability(0),
      "SMs", torch.cuda.get_device_properties(0).multi_processor_count)
DEVICE = torch.device("cuda")
T4_FP16_PEAK = 65e12  # FP16 tensor, mixed precision (T4 datasheet)

## The kernel (tutorial 03, adapted)

Each program computes one `BLOCK_SIZE_M × BLOCK_SIZE_N` tile of C. The first lines map the flat program id to a tile in *grouped* order; with `GROUP_SIZE_M = 1` the same code gives plain row-major order.

In [ ]:
@triton.jit
def leaky_relu(x):
    return tl.where(x >= 0, x, 0.01 * x)


@triton.jit
def matmul_kernel_raw(a_ptr, b_ptr, c_ptr, M, N, K,
                      stride_am, stride_ak, stride_bk, stride_bn, stride_cm, stride_cn,
                      BLOCK_SIZE_M: tl.constexpr, BLOCK_SIZE_N: tl.constexpr, BLOCK_SIZE_K: tl.constexpr,
                      GROUP_SIZE_M: tl.constexpr, ACTIVATION: tl.constexpr):
    # pid -> (pid_m, pid_n) in grouped order ("L2 Cache Optimizations" in the tutorial)
    pid = tl.program_id(axis=0)
    num_pid_m = tl.cdiv(M, BLOCK_SIZE_M)
    num_pid_n = tl.cdiv(N, BLOCK_SIZE_N)
    num_pid_in_group = GROUP_SIZE_M * num_pid_n
    group_id = pid // num_pid_in_group
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_SIZE_M)
    pid_m = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
    pid_n = (pid % num_pid_in_group) // group_size_m

    # pointer blocks for the first K step: &X[i, j] = X + i*stride_xi + j*stride_xj
    offs_am = (pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)) % M
    offs_bn = (pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)) % N
    offs_k = tl.arange(0, BLOCK_SIZE_K)
    a_ptrs = a_ptr + (offs_am[:, None] * stride_am + offs_k[None, :] * stride_ak)
    b_ptrs = b_ptr + (offs_k[:, None] * stride_bk + offs_bn[None, :] * stride_bn)

    accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)
    for k in range(0, tl.cdiv(K, BLOCK_SIZE_K)):
        # K-tail mask: out-of-range K columns load as 0.0 and add nothing
        a = tl.load(a_ptrs, mask=offs_k[None, :] < K - k * BLOCK_SIZE_K, other=0.0)
        b = tl.load(b_ptrs, mask=offs_k[:, None] < K - k * BLOCK_SIZE_K, other=0.0)
        accumulator = tl.dot(a, b, accumulator)
        a_ptrs += BLOCK_SIZE_K * stride_ak
        b_ptrs += BLOCK_SIZE_K * stride_bk
    # epilogue fused while the accumulator is still fp32
    if ACTIVATION == "leaky_relu":
        accumulator = leaky_relu(accumulator)
    c = accumulator.to(tl.float16)

    offs_cm = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
    offs_cn = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    c_ptrs = c_ptr + stride_cm * offs_cm[:, None] + stride_cn * offs_cn[None, :]
    c_mask = (offs_cm[:, None] < M) & (offs_cn[None, :] < N)
    tl.store(c_ptrs, c, mask=c_mask)


def C(bm, bn, bk, st, nw, g=8):
    return triton.Config({'BLOCK_SIZE_M': bm, 'BLOCK_SIZE_N': bn, 'BLOCK_SIZE_K': bk, 'GROUP_SIZE_M': g},
                         num_stages=st, num_warps=nw)

# The tutorial's 16 CUDA configs. Several need more shared memory than a T4 SM has (64 KB);
# the autotuner times those as failures and skips them.
CONFIGS = [C(128,256,64,3,8), C(64,256,32,4,4), C(128,128,32,4,4), C(128,64,32,4,4),
           C(64,128,32,4,4), C(128,32,32,4,4), C(64,32,32,5,2), C(32,64,32,5,2),
           C(128,256,128,3,8), C(256,128,128,3,8), C(256,64,128,4,4), C(64,256,128,4,4),
           C(128,128,128,4,4), C(128,64,64,4,4), C(64,128,64,4,4), C(128,32,64,4,4)]

matmul_kernel = triton.autotune(configs=CONFIGS, key=['M', 'N', 'K'])(matmul_kernel_raw)


def matmul(a, b, activation=""):
    assert a.shape[1] == b.shape[0] and a.is_contiguous()
    M, K = a.shape
    _, N = b.shape
    c = torch.empty((M, N), device=a.device, dtype=torch.float16)
    grid = lambda META: (triton.cdiv(M, META['BLOCK_SIZE_M']) * triton.cdiv(N, META['BLOCK_SIZE_N']),)
    matmul_kernel[grid](a, b, c, M, N, K, a.stride(0), a.stride(1), b.stride(0), b.stride(1),
                        c.stride(0), c.stride(1), ACTIVATION=activation)
    return c


def matmul_fixed(a, b, cfg, group_size_m):
    # same kernel, one forced config, no autotuning
    M, K = a.shape
    _, N = b.shape
    c = torch.empty((M, N), device=a.device, dtype=torch.float16)
    kw = dict(cfg.kwargs)
    kw['GROUP_SIZE_M'] = group_size_m
    grid = (triton.cdiv(M, kw['BLOCK_SIZE_M']) * triton.cdiv(N, kw['BLOCK_SIZE_N']),)
    matmul_kernel_raw[grid](a, b, c, M, N, K, a.stride(0), a.stride(1), b.stride(0), b.stride(1),
                            c.stride(0), c.stride(1), ACTIVATION="",
                            num_warps=cfg.num_warps, num_stages=cfg.num_stages, **kw)
    return c

## 1. Unit test against torch (cuBLAS)

The tutorial's test (512², `atol=1e-2`), plus an awkward shape (1000 × 1000 × 1000 is not a multiple of any block size, so the `% M`, `% N` wrap-around and the K-tail mask both matter), plus the fused leaky_relu. The first call triggers autotuning, so it takes a while.

In [ ]:
torch.manual_seed(0)
def check(M, N, K, activation="", atol=1e-2):
    a = torch.rand((M, K), device=DEVICE, dtype=torch.float16) - 0.5
    b = torch.rand((K, N), device=DEVICE, dtype=torch.float16) - 0.5
    out = matmul(a, b, activation)
    ref = torch.matmul(a, b)
    if activation == "leaky_relu":
        ref = torch.nn.functional.leaky_relu(ref.float(), 0.01).half()
    err = (out.float() - ref.float()).abs().max().item()
    ok = torch.allclose(out, ref, atol=atol, rtol=0)
    print(f"{M}x{K} @ {K}x{N} {activation or 'plain':<10} max|diff| = {err:.4f}  {'MATCH' if ok else 'DIFFER'}")
    return ok

results = [check(512, 512, 512), check(1000, 1000, 1000), check(512, 512, 512, "leaky_relu")]
assert all(results), "Triton and torch differ"

## 2. Benchmark: Triton vs `torch.matmul`, fp16, square sizes

`do_bench` runs warm-up and repetitions for fixed times (25 ms and 100 ms by default) and flushes L2 between runs; we take the median. TFLOPS = 2·M·N·K / time.

In [ ]:
def tflops(M, N, K, ms):
    return 2 * M * N * K * 1e-12 / (ms * 1e-3)

rows = []
for n in (1024, 2048, 4096):
    a = torch.randn((n, n), device=DEVICE, dtype=torch.float16)
    b = torch.randn((n, n), device=DEVICE, dtype=torch.float16)
    ms_t = triton.testing.do_bench(lambda: torch.matmul(a, b), quantiles=[0.5, 0.2, 0.8])[0]
    ms_tr = triton.testing.do_bench(lambda: matmul(a, b), quantiles=[0.5, 0.2, 0.8])[0]
    rows.append((n, tflops(n, n, n, ms_t), tflops(n, n, n, ms_tr)))

print(f"{'size':>6} {'torch/cuBLAS':>13} {'Triton':>8} {'Triton/cuBLAS':>14} {'Triton % of 65 TF':>18}")
for n, t, tr in rows:
    print(f"{n:>6} {t:>10.1f} TF {tr:>5.1f} TF {tr/t:>13.0%} {tr*1e12/T4_FP16_PEAK:>17.0%}")
print("Reference (tutorial page, GPU unnamed): 1024 104.9/95.3, 2048 217.9/184.4, 4096 221.1/219.7 TFLOPS")

## 3. Which config did the autotuner pick?

The autotuner benchmarks every config the first time it sees a new `(M, N, K)` (the `key`) and caches the winner. `.cache` maps each key to its chosen config.

In [ ]:
cache = getattr(matmul_kernel, "cache", {})
for key, cfg in cache.items():
    print(key, "->", cfg)
if not cache:
    print("best_config:", getattr(matmul_kernel, "best_config", "not exposed by this Triton version"))

## 4. GROUP_SIZE_M = 1 (row-major) vs grouped

We take the autotuner's pick for each shape and change only `GROUP_SIZE_M`. The card's calculator (`labs/grouped-ordering.py`) predicts how many distinct blocks the programs running at once touch; here you see whether that shows up as time on a T4 (40 SMs). The second shape is the Llama-3.1-8B MLP up-projection during a 4,096-token prefill, in fp16 instead of BF16.

In [ ]:
def best_cfg_for(M, N, K):
    a = torch.randn((M, K), device=DEVICE, dtype=torch.float16)
    b = torch.randn((K, N), device=DEVICE, dtype=torch.float16)
    matmul(a, b)  # makes sure this key has been tuned
    for key, cfg in getattr(matmul_kernel, "cache", {}).items():
        if tuple(key[:3]) == (M, N, K):
            return a, b, cfg
    return a, b, matmul_kernel.best_config

for (M, N, K) in [(4096, 4096, 4096), (4096, 14336, 4096)]:
    a, b, cfg = best_cfg_for(M, N, K)
    ref = torch.matmul(a, b)
    print(f"\n{M}x{K} @ {K}x{N}, base config {cfg}")
    base = None
    for g in (1, 2, 4, 8, 16):
        out = matmul_fixed(a, b, cfg, g)
        assert torch.allclose(out, ref, atol=1e-1, rtol=1e-2), f"mismatch at GROUP_SIZE_M={g}"
        ms = triton.testing.do_bench(lambda: matmul_fixed(a, b, cfg, g), quantiles=[0.5, 0.2, 0.8])[0]
        tf = tflops(M, N, K, ms)
        base = base or tf
        print(f"  GROUP_SIZE_M={g:>2}{' (row-major)' if g == 1 else '            '}  {ms:7.3f} ms  {tf:5.1f} TF"
              f"  {tf/base:5.2f}x vs row-major  {tf*1e12/T4_FP16_PEAK:4.0%} of 65 TF")

## Reference numbers and things to try

From Triton tutorial 03's rendered page (GPU not named on the page), fp16 TFLOPS, cuBLAS / Triton:

| size | cuBLAS | Triton |
|---|---|---|
| 1024 | 104.9 | 95.3 |
| 2048 | 217.9 | 184.4 |
| 4096 | 221.1 | 219.7 |

fp8 Triton at 4096: 206.2 TFLOPS. The tutorial also says grouped ordering can improve performance "by more than 10% on some hardware architecture (e.g., 220 to 245 TFLOPS on A100)". siboehm tried the same idea (thread swizzling) in FP32 CUDA on an RTX A6000 and saw no gain, with L2 hit rate already around 80%.

Your T4 has 65 TFLOPS FP16 tensor peak, so expect a fraction of those numbers. Whether `GROUP_SIZE_M` matters on your T4 is exactly the open question this notebook answers.

**Try this**

1. Change the Llama shape to M = 16 (a decode batch): `num_pid_m` becomes 1, and every `GROUP_SIZE_M` gives the same order. Does time change at all?
2. Delete every config except `C(64,32,32,5,2)` and rerun section 2: how much does autotuning buy on each size?
3. In Nsight Compute (see the profiling card), compare `lts__t_sector_hit_rate.pct` (L2 hit rate) for `GROUP_SIZE_M=1` vs `8` on the 4096 × 14336 shape.